# 04 — Phase 4 : générer un plan de jeu avec Claude

Objectif : transformer l'attribution SHAP (phase 3) en plan de jeu rédigé, pour une composition hypothétique (deux équipes de 5 champions, pas un match réel).

Toute la logique de classement (`composition_fact_ranking`) a été **industrialisée** dans `lol_assistant/game_plan.py` — ce notebook l'utilise, il ne la redéfinit pas. C'est exactement pour ça qu'on avait fait ce travail d'industrialisation en phase 2 : la logique réutilisable vit dans le package, les notebooks orchestrent et expliquent.

## Concept 1 — Classer les faits pour les deux équipes

`load_ranking_artifacts` lit les deux CSV produits par le notebook 3 (`global_shap_importance.csv`, `champion_feature_interactions.csv`) — aucun appel au modèle XGBoost ni à SHAP ici, juste une lecture de fichiers, donc très rapide. `composition_fact_ranking` applique la même logique que dans le notebook 3 (le max par champion, pas la somme).

In [1]:
from lol_assistant.game_plan import load_ranking_artifacts, composition_fact_ranking

global_importance, interactions = load_ranking_artifacts()

your_team = ["Jayce", "Sylas", "Yone", "Yunara", "Lulu"]
enemy_team = ["Garen", "Rengar", "Akali", "Hwei", "Rakan"]

your_ranking = composition_fact_ranking(your_team, global_importance, interactions)
enemy_ranking = composition_fact_ranking(enemy_team, global_importance, interactions)

your_ranking.head(6)[["feature", "key_champion", "team_importance"]]

,feature,key_champion,team_importance
0,voidMonsterKill,Jayce,0.664870
1,firstTurretKilledTime,Yone,0.664481
2,maxLevelLeadLaneOpponent,Sylas,0.655693
3,earliestDragonTakedown,Lulu,0.451573
4,teamRiftHeraldKills,Yunara,0.279357
5,visionScoreAdvantageLaneOpponent,Lulu,0.258550


## Concept 2 — Construire le prompt

`build_game_plan_prompt` transforme les deux classements en texte : les faits de **notre** équipe ("à prioriser") et ceux de l'équipe **adverse** ("à surveiller/contrer" — même analyse légitime appliquée à leurs champions, pas un vrai signal de matchup entraîné, comme convenu). On limite volontairement aux 6 faits les plus importants par équipe (`top_n`) — inutile de noyer le LLM avec les 27, les derniers ont une importance quasi nulle de toute façon.

In [2]:
from lol_assistant.game_plan import build_game_plan_prompt

prompt = build_game_plan_prompt(your_team, enemy_team, your_ranking, enemy_ranking)
print(prompt)

Voici une composition League of Legends à analyser.

Notre équipe : Jayce, Sylas, Yone, Yunara, Lulu
Équipe adverse : Garen, Rengar, Akali, Hwei, Rakan

D'après un modèle entraîné sur des dizaines de milliers de vraies parties, voici les faits de jeu (objectifs, avance en lane, etc.) qui comptent le plus pour NOTRE équipe, avec le champion à qui chaque fait profite le plus (élément statistique, pas une garantie) :
- voidMonsterKill (surtout via Jayce)
- firstTurretKilledTime (surtout via Yone)
- maxLevelLeadLaneOpponent (surtout via Sylas)
- earliestDragonTakedown (surtout via Lulu)
- teamRiftHeraldKills (surtout via Yunara)
- visionScoreAdvantageLaneOpponent (surtout via Lulu)

Et voici les mêmes faits de jeu, mais pour l'équipe ADVERSE — donc ce sur quoi elle va probablement s'appuyer, et qu'on devrait chercher à empêcher ou retarder :
- voidMonsterKill (surtout via Garen)
- maxLevelLeadLaneOpponent (surtout via Akali)
- firstTurretKilledTime (surtout via Rakan)
- earliestDragonTaked

## Concept 3 — Appeler Claude

`generate_game_plan` fait tout le pipeline (classement + prompt + appel API) en un appel. On utilise **Claude Sonnet 5** par défaut (`DEFAULT_MODEL` dans `game_plan.py`) — tâche de rédaction assez "gabarisée" (données structurées → texte), pas besoin du modèle le plus capable ; passer `model="claude-opus-5"` pour la meilleure qualité possible si besoin.

**Nécessite `ANTHROPIC_API_KEY` renseignée dans `.env`** (comme pour Riot en phase 1).

In [3]:
from lol_assistant.game_plan import generate_game_plan

game_plan_text, your_ranking, enemy_ranking = generate_game_plan(your_team, enemy_team)
print(game_plan_text)

# Plan de jeu

**Notre priorité : imposer un rythme rapide et écraser la lane top/mid.**

Sylas doit chercher à dominer sa lane en level (grâce aux roams de Jayce ou aux ganks de Yone), pour transformer cet avantage en pression sur la carte entière. Une fois devant, il peut splitpush ou venir prendre des teamfights avec un ultime déjà chargé — c'est notre meilleur levier de snowball.

Jayce doit prioriser les monstres du Vide dès qu'ils apparaissent : il a le poke et la mobilité pour les sécuriser sans trop de risque, et ça donne un vrai coup d'accélérateur à l'équipe.

Yone doit viser la première tour très tôt — avec Yunara en bot lane qui peut aussi punir les invasions de Rift Herald adverses, on veut faire tomber les objectifs de structure avant que l'adversaire ne stabilise.

Lulu est notre clé vision/dragon : elle doit poser des wards tôt dans le jungle et autour de la rivière pour garantir le premier dragon et priver l'adversaire d'infos, tout en gardant l'avantage de vision sur 

---

**Checkpoint** : confirme que le plan de jeu généré est cohérent (mentionne bien les faits clés et les champions associés, pas de contresens). Prochaine étape : ajouter Ollama/Llama 3.1 comme option de comparaison (gratuite, locale).